# MOHIM raw + mean-centered motif dataset and stem-wise threshold diagnostics

음원을 source 단위로 한 번 분리해 dataset stem과 앞 30초의 모든 4마디 후보 진단 결과를 함께 저장합니다.

## 0. Drive와 mean-centering 브랜치 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'codex/occurrence'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repository: /content/MOHIM


In [ ]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. 실험 경로와 범위

In [ ]:
MAX_SONGS = 3000
DATA_SOURCE = 'billboard'  # 'songs' 또는 'billboard'
FORCE_REPROCESS = True
DEVICE = 'cuda'
DEMUCS_BATCH_SIZE = 8
MOTIF_WORKERS = 12
DEMUCS_SHIFTS = 0
AUDIO_FORMAT = 'flac'
MOTIF_BARS = 4
MOTIF_SEARCH_SECONDS = 30.0
MIN_ACTIVE_RATIO = 0.65
MAX_ONSET_CHROMA_DIFFERENCE = 0.40
MIN_ONSET_SIMILARITY = 0.54
MIN_PITCH_CLASS_SPAN = 0.25
MIN_MEAN_CENTERED_SIMILARITY = 0.25
MIN_ONSET_VARIATION = 0.25
SIMILARITY_MODE = 'raw_gate_mean_centered_rank_v1'
MIN_NEXT_BAR_SCORE_GAIN = 0.05

BAR_ALIGNMENT_SCORE_KEYS = (
    'onset_similarity',
    'chroma_similarity',
    'mean_centered_onset_similarity',
    'mean_centered_chroma_similarity',
)

SONGS_DIR = Path('/content/drive/MyDrive/MOHIM/songs')
BILLBOARD_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/billboard_pop_dataset')
MOTIF_DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MOTIF_VERSION_PATH = MOTIF_DATASET_DIR / '.mohim_motif_version'
SONGS_STEM_DIR = Path('/content/drive/MyDrive/MOHIM/songs_stem_dataset')
DIAGNOSTIC_DIR = Path('/content/drive/MyDrive/MOHIM/motif_stem_diagnostics_mean_centered')
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')
AUDIO_EXTENSIONS = {'.aac', '.flac', '.m4a', '.mp3', '.ogg', '.wav', '.webm'}

assert DATA_SOURCE in {'songs', 'billboard'}
MOTIF_DATASET_DIR.mkdir(parents=True, exist_ok=True)
SONGS_STEM_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

## 1-1. `songs` 폴더 입력

`DATA_SOURCE = 'songs'`일 때만 실행되며 파일명을 곡 제목으로 사용합니다.

In [ ]:
if DATA_SOURCE == 'songs':
    song_paths = sorted(
        path for path in SONGS_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )[:MAX_SONGS]
    songs = [
        {
            'track_id': path.stem,
            'artist': '',
            'title': path.stem,
            'audio_path': path,
        }
        for path in song_paths
    ]

## 1-2. Billboard dataset 입력

`DATA_SOURCE = 'billboard'`일 때 `billboard_pop_dataset/tracks.json`과 `audio` 폴더에서 실제 음원이 있는 곡을 최대 10개 가져옵니다.

In [ ]:
if DATA_SOURCE == 'billboard':
    from mohim.dataset import index_audio_files, load_local_tracks, resolve_audio_path

    tracks_json = BILLBOARD_DATASET_DIR / 'tracks.json'
    audio_dir = BILLBOARD_DATASET_DIR / 'audio'
    assert tracks_json.is_file(), f'tracks.json이 없습니다: {tracks_json}'
    assert audio_dir.is_dir(), f'음원 폴더가 없습니다: {audio_dir}'

    tracks = load_local_tracks(tracks_json, require_lyrics=True)
    audio_index = index_audio_files(audio_dir)
    songs = []
    for track in tracks:
        audio_path = resolve_audio_path(track, audio_index)
        if audio_path is None:
            print(f'[skip] 음원 없음: {track.artist} - {track.title}')
            continue
        songs.append({
            'track_id': track.track_id,
            'artist': track.artist,
            'title': track.title,
            'audio_path': audio_path,
            'track': track,
        })
        if len(songs) >= MAX_SONGS:
            break
    song_paths = [song['audio_path'] for song in songs]

## 1-3. 선택한 입력 확인

In [ ]:
assert songs, f'사용 가능한 음원이 없습니다: {DATA_SOURCE}'
assert len(songs) == len(song_paths)
print(f'data source: {DATA_SOURCE}')
print('songs:', len(songs))
for song in songs:
    label = f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    print('-', label)

data source: billboard
songs: 2129
- 2 Pistols Featuring T-Pain & Tay Dizm - She Got It
- 21 Savage - Redrum
- 2AM Club - Worry About You
- 3 Doors Down - It's Not My Time
- 3OH!3 - Don't Trust Me
- 3OH!3 - Double Vision
- 3OH!3 Featuring Ke$ha - My First Kiss
- 5 Seconds of Summer - Amnesia
- 5 Seconds of Summer - Complete Mess
- 5 Seconds of Summer - Easier
- 5 Seconds of Summer - Hey Everybody!
- 5 Seconds of Summer - Me Myself & I
- 5 Seconds of Summer - Old Me
- 5 Seconds of Summer - She Looks So Perfect
- 5 Seconds of Summer - She's Kinda Hot
- 5 Seconds of Summer - Teeth
- 5 Seconds of Summer - Want You Back
- 5 Seconds of Summer - What I Like About You
- 5 Seconds of Summer - Youngblood
- 50 Cent Featuring Justin Timberlake & Timbaland - Ayo Technology
- 50 Cent Featuring Eminem & Adam Levine - My Life
- 88rising & Joji & Jackson Wang Featuring Swae Lee &Major Lazer - Walking
- A Boogie wit da Hoodie - Look Back At It
- A Great Big World & Christina Aguilera - Say Something
- A

## 2. Demucs와 motif scorer 준비

In [ ]:
import urllib.request
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker, onset_variation
from mohim.dataset import DatasetBuilder
from mohim.separator import StemSeparator

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt', BEAT_CHECKPOINT
    )
separator = StemSeparator(
    device=DEVICE, model_name='htdemucs_6s', shifts=DEMUCS_SHIFTS
)
beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_scorer = MotifExtractor(
    beat_tracker,
    MotifConfig(
        bars=MOTIF_BARS,
        search_seconds=MOTIF_SEARCH_SECONDS,
        min_presence=MIN_ACTIVE_RATIO,
        max_similarity_difference=MAX_ONSET_CHROMA_DIFFERENCE,
        onset_threshold=MIN_ONSET_SIMILARITY,
        pitch_class_span_threshold=MIN_PITCH_CLASS_SPAN,
        mean_centered_similarity_threshold=MIN_MEAN_CENTERED_SIMILARITY,
        onset_variation_threshold=MIN_ONSET_VARIATION,
    ),
)

dataset_builder = (
    DatasetBuilder(
        audio_dir=BILLBOARD_DATASET_DIR / 'audio',
        output_dir=MOTIF_DATASET_DIR,
        separator=separator,
        motif_extractor=motif_scorer,
        audio_format=AUDIO_FORMAT,
        resume=not FORCE_REPROCESS,
    )
    if DATA_SOURCE == 'billboard' else None
)
print('separator and motif scorer ready')

separator and motif scorer ready


## 3. 모든 stem × start-downbeat 후보 계산

`DEMUCS_BATCH_SIZE`개 곡을 한 GPU batch로 분리하고, 해당 batch의 곡 × stem 점수 계산은 `MOTIF_WORKERS`개 CPU thread로 병렬 실행합니다. Raw onset/chroma는 1차 후보 필터에 사용하고, mean-centered onset/chroma는 stem별 최초 후보의 최종 순위와 0.28 하한 검사에 사용합니다. Onset 하한에 미달한 후보는 pitch-class span 계산을 생략하며, `candidate_*.flac`은 active ratio, raw onset/chroma 차이, raw onset, pitch-class span 조건까지 통과한 구간만 저장합니다.

In [ ]:
from collections import Counter
import json
import librosa
import numpy as np
import pandas as pd
import re
import shutil
from mohim.separator import save_audio

def average_next_bar_score_gain(current, next_candidate):
    """
    현재 후보보다 다음 마디 후보의 4개 점수가
    평균적으로 얼마나 좋아졌는지 계산.
    """
    gains = [
        next_candidate[key] - current[key]
        for key in BAR_ALIGNMENT_SCORE_KEYS
    ]

    return float(np.mean(gains))

def safe_folder_name(value):
    cleaned = re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_', value)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip(' .')
    return cleaned[:120] or 'untitled'

def song_display_title(song):
    return f"{song['artist']} - {song['title']}" if song['artist'] else song['title']

diagnostic_base_name_counts = Counter(
    safe_folder_name(song_display_title(song)) for song in songs
)

def diagnostic_song_id(song):
    base_name = safe_folder_name(song_display_title(song))
    if diagnostic_base_name_counts[base_name] == 1:
        return base_name
    short_track_id = safe_folder_name(str(song['track_id']))[:8]
    return f'{base_name[:109]} [{short_track_id}]'

def song_stem_metadata_path(song):
    display_title = song_display_title(song)
    return SONGS_STEM_DIR / safe_folder_name(display_title) / 'metadata.json'

def song_stems_complete(song, audio_path):
    metadata_path = song_stem_metadata_path(song)
    if not metadata_path.is_file():
        return False
    try:
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        same_audio = Path(metadata.get('source_audio', '')).resolve() == Path(audio_path).resolve()
        stem_files = metadata.get('stem_files')
        return (
            same_audio
            and isinstance(stem_files, dict)
            and stem_files
            and all((metadata_path.parent / filename).is_file() for filename in stem_files.values())
        )
    except (OSError, ValueError, TypeError):
        return False

def save_song_stems(song, audio_path, stems, sample_rate):
    metadata_path = song_stem_metadata_path(song)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    stem_files = {}
    for stem_name, stem_audio in stems.items():
        if not stem_name.replace('_', '').isalnum():
            raise ValueError(f'Unsafe separator stem name: {stem_name!r}')
        filename = f'{stem_name}.{AUDIO_FORMAT}'
        save_audio(metadata_path.parent / filename, stem_audio, sample_rate, audio_format=AUDIO_FORMAT)
        stem_files[stem_name] = filename
    metadata = {
        'track_id': song['track_id'], 'artist': song['artist'], 'title': song['title'],
        'source_audio': str(audio_path), 'sample_rate': sample_rate, 'stem_files': stem_files,
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return metadata_path.parent

def internal_candidate_features(audio_path):
    audio, sample_rate = librosa.load(audio_path, sr=None, mono=True)
    hop_length = 512
    chroma = librosa.feature.chroma_cens(
        y=audio, sr=sample_rate, hop_length=hop_length
    )
    chroma_sum = chroma.sum(axis=0, keepdims=True)
    valid_chroma = chroma_sum.ravel() > 1e-8
    chroma_norm = np.divide(
        chroma, chroma_sum, out=np.zeros_like(chroma), where=chroma_sum > 1e-8
    )
    valid_pairs = valid_chroma[:-1] & valid_chroma[1:]
    if np.any(valid_pairs):
        frame_flux = 0.5 * np.sum(np.abs(np.diff(chroma_norm, axis=1)), axis=0)
        chroma_flux = float(np.mean(frame_flux[valid_pairs]))
    else:
        chroma_flux = 0.0
    return {'chroma_flux': chroma_flux}

def candidate_passes_export_filters(row):
    return (
        row['active_ratio'] >= MIN_ACTIVE_RATIO
        and abs(row['onset_similarity'] - row['chroma_similarity']) <= MAX_ONSET_CHROMA_DIFFERENCE
        and row['onset_similarity'] >= MIN_ONSET_SIMILARITY
        and row['pitch_class_span'] is not None
        and row['pitch_class_span'] >= MIN_PITCH_CLASS_SPAN
    )

motif_result_counts = {'successed': 0, 'failed': 0}

def print_motif_result(title, succeeded, detail=None):
    result_key = 'successed' if succeeded else 'failed'
    motif_result_counts[result_key] += 1
    label = 'selection' if succeeded else 'skipped'
    counts = (
        f"successed: {motif_result_counts['successed']} / "
        f"failed: {motif_result_counts['failed']}"
    )
    detail_suffix = f' - {detail}' if detail else ''
    print(f'  [{label}] {title} ({counts}){detail_suffix}')

MIN_NEXT_BAR_SCORE_GAIN = 0.05

BAR_ALIGNMENT_SCORE_KEYS = (
    'onset_similarity',
    'chroma_similarity',
    'mean_centered_onset_similarity',
    'mean_centered_chroma_similarity',
)


def average_next_bar_score_gain(current, next_candidate):
    """
    현재 후보와 다음 마디 후보의 4개 점수 차이(next - current)를
    평균내서 다음 마디로 이동할지 판단.
    """
    return float(np.mean([
        next_candidate[key] - current[key]
        for key in BAR_ALIGNMENT_SCORE_KEYS
    ]))


def save_earliest_valid_selection(metadata_path, metadata):
    # 이전 selection 파일 제거
    for stale_name in (
        'motif_selections.json',
        'selected_earliest.flac',
        'selected_highest_similarity.flac',
    ):
        stale_path = metadata_path.parent / stale_name
        if stale_path.is_file():
            stale_path.unlink()

    # =========================================================
    # 1. 1차 유효성 필터
    #
    # 여기서는:
    #   - active_ratio
    #   - onset_similarity
    #   - onset/chroma difference
    #   - pitch_class_span
    #
    # 만 검사.
    #
    # mean-centered / onset_variation은 아직 검사하지 않음.
    # =========================================================
    eligible = [
        row
        for row in metadata['candidates']
        if candidate_passes_export_filters(row)
    ]

    if not eligible:
        skipped_metadata = {
            'song_id': metadata['song_id'],
            'title': metadata['title'],
            'source_audio': metadata['source_audio'],
            'status': 'skipped',
            'reason': 'no_candidate_passed_export_filters',

            'min_active_ratio': MIN_ACTIVE_RATIO,
            'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
            'onset_threshold': MIN_ONSET_SIMILARITY,
            'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
            'mean_centered_similarity_threshold': MIN_MEAN_CENTERED_SIMILARITY,
            'onset_variation_threshold': MIN_ONSET_VARIATION,
            'next_bar_score_gain_threshold': MIN_NEXT_BAR_SCORE_GAIN,

            'similarity_mode': SIMILARITY_MODE,
            'candidate_count': len(metadata['candidates']),
            'eligible_candidate_count': 0,

            'alignment_decisions': {},
            'selected_by_stem_before_final_check': {},
            'final_validation': {},
            'selections': {},
        }

        (metadata_path.parent / 'motif_selections.json').write_text(
            json.dumps(
                skipped_metadata,
                ensure_ascii=False,
                indent=2,
            ),
            encoding='utf-8',
        )

        print_motif_result(
            metadata['title'],
            False,
            '1차 조건 통과 후보 없음',
        )

        return skipped_metadata

    # =========================================================
    # 2. stem별로 1차 유효 후보 묶기
    # =========================================================
    candidates_by_stem = {}

    for row in eligible:
        candidates_by_stem.setdefault(
            row['stem_name'],
            [],
        ).append(row)

    # 각 stem 내부는 시간순 정렬
    for stem_name in candidates_by_stem:
        candidates_by_stem[stem_name].sort(
            key=lambda row: row['start_sec']
        )

    # =========================================================
    # 3. stem 내부 시작점 결정
    #
    # 가장 이른 1차 유효 후보부터 시작.
    #
    # current → next의 4개 점수 평균 개선량이
    # 0.05 이상이면 다음 마디로 이동.
    #
    # 이 단계에서는 최종 threshold를 검사하지 않음.
    # =========================================================
    selected_by_stem_before_final_check = {}
    alignment_decisions = {}

    for stem_name, stem_rows in candidates_by_stem.items():
        current_pos = 0
        current = stem_rows[current_pos]

        decisions = []

        while current_pos + 1 < len(stem_rows):
            next_pos = current_pos + 1
            next_candidate = stem_rows[next_pos]

            score_differences = {
                key: float(
                    next_candidate[key] - current[key]
                )
                for key in BAR_ALIGNMENT_SCORE_KEYS
            }

            average_gain = float(np.mean(
                list(score_differences.values())
            ))

            decision = {
                'current_candidate_index': current['candidate_index'],
                'current_start_sec': current['start_sec'],

                'next_candidate_index': next_candidate['candidate_index'],
                'next_start_sec': next_candidate['start_sec'],

                'score_differences': score_differences,
                'average_score_gain': average_gain,
                'threshold': MIN_NEXT_BAR_SCORE_GAIN,
            }

            # ---------------------------------------------
            # 다음 마디가 평균적으로 0.05 이상 더 좋으면 이동
            # ---------------------------------------------
            if average_gain >= MIN_NEXT_BAR_SCORE_GAIN:
                decision['action'] = 'move_to_next_bar'
                decisions.append(decision)

                current_pos = next_pos
                current = next_candidate

                # 이동했으므로 새 current와 그 다음 후보를 다시 비교
                continue

            # ---------------------------------------------
            # 충분한 차이가 아니면 현재 시작점 유지
            # ---------------------------------------------
            decision['action'] = 'keep_current'
            decisions.append(decision)

            break

        selected_by_stem_before_final_check[stem_name] = dict(current)

        alignment_decisions[stem_name] = {
            'initial_candidate_index': stem_rows[0]['candidate_index'],
            'initial_start_sec': stem_rows[0]['start_sec'],

            'selected_candidate_index': current['candidate_index'],
            'selected_start_sec': current['start_sec'],

            'decisions': decisions,
        }

    # =========================================================
    # 4. 이제 stem별로 선택된 후보에 대해서만 최종 점검
    #
    #   - mean-centered onset similarity
    #   - mean-centered chroma similarity
    #   - onset variation
    #
    # 여기서 실패하면 그 stem은 최종 후보에서 탈락.
    # 다른 candidate로 fallback하지 않음.
    # =========================================================
    final_valid_by_stem = {}
    final_validation = {}

    for stem_name, selected_row in (
        selected_by_stem_before_final_check.items()
    ):
        row = dict(selected_row)

        validation = {
            'candidate_index': row['candidate_index'],
            'start_sec': row['start_sec'],

            'mean_centered_onset_similarity':
                row['mean_centered_onset_similarity'],

            'mean_centered_chroma_similarity':
                row['mean_centered_chroma_similarity'],
        }

        # ---------------------------------------------
        # 4-1. mean-centered similarity 최종 검사
        # ---------------------------------------------
        mean_centered_onset_pass = (
            row['mean_centered_onset_similarity']
            >= MIN_MEAN_CENTERED_SIMILARITY
        )

        mean_centered_chroma_pass = (
            row['mean_centered_chroma_similarity']
            >= MIN_MEAN_CENTERED_SIMILARITY
        )

        validation['mean_centered_onset_pass'] = (
            mean_centered_onset_pass
        )

        validation['mean_centered_chroma_pass'] = (
            mean_centered_chroma_pass
        )

        if not (
            mean_centered_onset_pass
            and mean_centered_chroma_pass
        ):
            validation['passed'] = False
            validation['reason'] = (
                'mean_centered_similarity_below_threshold'
            )

            final_validation[stem_name] = validation
            continue

        # ---------------------------------------------
        # 4-2. onset variation 최종 검사
        # ---------------------------------------------
        candidate_path = (
            metadata_path.parent
            / row['candidate_file']
        )

        candidate_audio, candidate_sample_rate = librosa.load(
            candidate_path,
            sr=None,
            mono=False,
        )

        variation = onset_variation(
            candidate_audio,
            candidate_sample_rate,
        )

        row['onset_variation'] = variation

        validation['onset_variation'] = variation
        validation['onset_variation_pass'] = (
            variation >= MIN_ONSET_VARIATION
        )

        if variation < MIN_ONSET_VARIATION:
            validation['passed'] = False
            validation['reason'] = (
                'onset_variation_below_threshold'
            )

            final_validation[stem_name] = validation
            continue

        # ---------------------------------------------
        # 최종 통과
        # ---------------------------------------------
        validation['passed'] = True
        validation['reason'] = None

        final_validation[stem_name] = validation
        final_valid_by_stem[stem_name] = row

    # =========================================================
    # 5. 모든 stem이 최종 검사에서 탈락한 경우
    # =========================================================
    if not final_valid_by_stem:
        skipped_metadata = {
            'song_id': metadata['song_id'],
            'title': metadata['title'],
            'source_audio': metadata['source_audio'],
            'status': 'skipped',
            'reason': 'no_stem_candidate_passed_final_validation',

            'min_active_ratio': MIN_ACTIVE_RATIO,
            'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
            'onset_threshold': MIN_ONSET_SIMILARITY,
            'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
            'mean_centered_similarity_threshold':
                MIN_MEAN_CENTERED_SIMILARITY,
            'onset_variation_threshold': MIN_ONSET_VARIATION,
            'next_bar_score_gain_threshold': MIN_NEXT_BAR_SCORE_GAIN,

            'bar_alignment_score_keys':
                list(BAR_ALIGNMENT_SCORE_KEYS),

            'similarity_mode': SIMILARITY_MODE,

            'candidate_count': len(metadata['candidates']),
            'eligible_candidate_count': len(eligible),

            'alignment_decisions': alignment_decisions,
            'selected_by_stem_before_final_check':
                selected_by_stem_before_final_check,
            'final_validation': final_validation,

            'selections': {},
        }

        (metadata_path.parent / 'motif_selections.json').write_text(
            json.dumps(
                skipped_metadata,
                ensure_ascii=False,
                indent=2,
            ),
            encoding='utf-8',
        )

        print_motif_result(
            metadata['title'],
            False,
            'stem별 시작점 결정 후 최종 검증 통과 후보 없음',
        )

        return skipped_metadata

    # =========================================================
    # 6. 최종 검증까지 통과한 stem 후보들 중
    #    가장 이른 start_sec 선택
    # =========================================================
    earliest = dict(
        min(
            final_valid_by_stem.values(),
            key=lambda row: row['start_sec'],
        )
    )

    selection_file = 'selected_earliest.flac'

    shutil.copy2(
        metadata_path.parent / earliest['candidate_file'],
        metadata_path.parent / selection_file,
    )

    earliest['selection_file'] = selection_file

    # =========================================================
    # 7. selection metadata 저장
    # =========================================================
    selection_metadata = {
        'song_id': metadata['song_id'],
        'title': metadata['title'],
        'source_audio': metadata['source_audio'],
        'status': 'selected',

        'min_active_ratio': MIN_ACTIVE_RATIO,
        'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
        'onset_threshold': MIN_ONSET_SIMILARITY,
        'pitch_class_span_threshold': MIN_PITCH_CLASS_SPAN,
        'mean_centered_similarity_threshold':
            MIN_MEAN_CENTERED_SIMILARITY,
        'onset_variation_threshold': MIN_ONSET_VARIATION,

        'next_bar_score_gain_threshold': MIN_NEXT_BAR_SCORE_GAIN,
        'bar_alignment_score_keys': list(BAR_ALIGNMENT_SCORE_KEYS),

        'similarity_mode': SIMILARITY_MODE,

        'candidate_count': len(metadata['candidates']),
        'eligible_candidate_count': len(eligible),

        # 시작점 보정 과정
        'alignment_decisions': alignment_decisions,

        # 최종 검증 전 stem별 선택 결과
        'selected_by_stem_before_final_check':
            selected_by_stem_before_final_check,

        # 최종 검증 결과
        'final_validation': final_validation,

        # 최종 검증까지 통과한 stem들
        'final_valid_by_stem': final_valid_by_stem,

        # 곡 전체 최종 선택
        'selections': {
            'earliest': earliest,
        },
    }

    (metadata_path.parent / 'motif_selections.json').write_text(
        json.dumps(
            selection_metadata,
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )

    print_motif_result(
        metadata['title'],
        True,
        (
            f"{earliest['stem_name']} "
            f"@ {earliest['start_sec']:.2f}s"
        ),
    )

    return selection_metadata

all_rows = []
dataset_results = []
audio_failures = []
pending_jobs = []

def is_audio_decode_error(error):
    decode_markers = ('Failed to decode audio samples', 'Could not push packet', 'Invalid data found')
    seen = set()
    while error is not None and id(error) not in seen:
        if any(marker in str(error) for marker in decode_markers):
            return True
        seen.add(id(error))
        error = error.__cause__ or error.__context__
    return False

assert DEMUCS_BATCH_SIZE >= 1
assert MOTIF_WORKERS >= 1
for song_index, audio_path in enumerate(song_paths):
    song = songs[song_index]
    display_title = song_display_title(song)
    print(f'[{song_index + 1}/{len(song_paths)}] {display_title}')
    song_id = diagnostic_song_id(song)
    song_dir = DIAGNOSTIC_DIR / song_id
    if not FORCE_REPROCESS and not song_dir.exists():
        legacy_song_dir = DIAGNOSTIC_DIR / f'{song_index:03d}_{safe_folder_name(display_title)}'
        if legacy_song_dir.exists():
            legacy_song_dir.rename(song_dir)
            print(f'  [rename] {legacy_song_dir.name} -> {song_dir.name}')
    metadata_path = song_dir / 'motif_scores.json'

    existing = None
    if not FORCE_REPROCESS and metadata_path.is_file() and (song_dir / 'melodic_accompaniment.flac').is_file():
        try:
            candidate_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            same_audio = (
                Path(candidate_metadata.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            same_similarity_mode = candidate_metadata.get('similarity_mode') == SIMILARITY_MODE
            candidates = candidate_metadata.get('candidates')
            candidates_complete = isinstance(candidates, list) and all(
                'pitch_class_span' in row
                and 'mean_centered_onset_similarity' in row
                and 'mean_centered_chroma_similarity' in row
                and (
                    not candidate_passes_export_filters(row)
                    or (
                        row.get('candidate_file')
                        and (song_dir / row['candidate_file']).is_file()
                    )
                )
                for row in candidate_metadata.get('candidates', [])
            )
            if same_audio and same_similarity_mode and candidates_complete:
                existing = candidate_metadata
        except (OSError, ValueError, TypeError):
            existing = None

    if existing is not None:
        selection_path = song_dir / 'motif_selections.json'
        try:
            cached_selection = json.loads(selection_path.read_text(encoding='utf-8'))
        except (OSError, ValueError, TypeError):
            cached_selection = None
        if cached_selection is not None and cached_selection.get('status') == 'skipped':
            same_skip_audio = (
                Path(cached_selection.get('source_audio', '')).resolve()
                == Path(audio_path).resolve()
            )
            same_skip_config = all((
                cached_selection.get('min_active_ratio') == MIN_ACTIVE_RATIO,
                cached_selection.get('max_onset_chroma_difference') == MAX_ONSET_CHROMA_DIFFERENCE,
                cached_selection.get('onset_threshold') == MIN_ONSET_SIMILARITY,
                cached_selection.get('pitch_class_span_threshold') == MIN_PITCH_CLASS_SPAN,
                cached_selection.get('mean_centered_similarity_threshold') == MIN_MEAN_CENTERED_SIMILARITY,
                cached_selection.get('onset_variation_threshold') == MIN_ONSET_VARIATION,
                cached_selection.get('similarity_mode') == SIMILARITY_MODE,
            ))
            if same_skip_audio and same_skip_config:
                for row in existing['candidates']:
                    all_rows.append({'song_id': song_id, 'title': display_title, **row})
                if dataset_builder is not None:
                    dataset_results.append(dataset_builder.record_rejection(
                        song['track'], cached_selection.get('reason', 'final_motif_rejected')
                    ))
                print_motif_result(
                    display_title, False, 'cached: 이전 실행에서 통과한 모티프 없음'
                )
                continue
            print('  [reprocess] 스킵 이후 음원 또는 임계값 변경')
            existing = None

    if existing is not None:
        selection_metadata = save_earliest_valid_selection(metadata_path, existing)
        if dataset_builder is not None:
            if selection_metadata['status'] == 'selected':
                selected_candidate = selection_metadata['selections']['earliest']
                validated_variation = selected_candidate['onset_variation']
                dataset_result = dataset_builder.process_track(
                    song['track'], audio_index,
                    validated_onset_variation=validated_variation,
                    selected_candidate=selected_candidate,
                )
            else:
                dataset_result = dataset_builder.record_rejection(
                    song['track'], selection_metadata.get('reason', 'final_motif_rejected')
                )
            dataset_results.append(dataset_result)
        elif not song_stems_complete(song, audio_path):
            print('  [process] 저장되지 않은 songs stem 분리')
            stems, sample_rate, mixture = separator.separate(audio_path)
            save_song_stems(song, audio_path, stems, sample_rate)
            del stems, mixture
        print('  [reuse] 기존 분리/후보 결과 사용')
        for row in existing['candidates']:
            all_rows.append({'song_id': song_id, 'title': display_title, **row})
        continue

    pending_jobs.append({
        'song_index': song_index, 'song': song, 'audio_path': audio_path,
        'display_title': display_title, 'song_id': song_id,
        'song_dir': song_dir, 'metadata_path': metadata_path,
    })

for batch_start in range(0, len(pending_jobs), DEMUCS_BATCH_SIZE):
    batch_jobs = pending_jobs[batch_start:batch_start + DEMUCS_BATCH_SIZE]
    batch_paths = [job['audio_path'] for job in batch_jobs]
    batch_number = batch_start // DEMUCS_BATCH_SIZE + 1
    batch_count = (len(pending_jobs) + DEMUCS_BATCH_SIZE - 1) // DEMUCS_BATCH_SIZE
    print(f'[Demucs batch {batch_number}/{batch_count}] {len(batch_jobs)}곡 분리')
    try:
        batch_separations = separator.separate_many(batch_paths)
    except RuntimeError as batch_error:
        if not is_audio_decode_error(batch_error):
            raise

        valid_jobs = []
        for job in batch_jobs:
            try:
                separator._load_audio(job['audio_path'])
            except Exception as audio_error:
                if not is_audio_decode_error(audio_error):
                    raise
                failure = {
                    'song_id': job['song_id'],
                    'title': job['display_title'],
                    'source_audio': str(job['audio_path']),
                    'status': 'failed',
                    'stage': 'audio_decode',
                    'error_type': type(audio_error).__name__,
                    'error_message': str(audio_error),
                }
                audio_failures.append(failure)
                print(f"  [skip: audio decode failed] {job['display_title']}: {audio_error}")
                if dataset_builder is not None:
                    dataset_results.append(dataset_builder.record_rejection(
                        job['song']['track'],
                        f"audio_decode_failed: {type(audio_error).__name__}: {audio_error}",
                    ))
            else:
                valid_jobs.append(job)

        batch_jobs = valid_jobs
        batch_paths = [job['audio_path'] for job in batch_jobs]
        batch_separations = separator.separate_many(batch_paths) if batch_paths else []
    assert len(batch_separations) == len(batch_jobs)
    batch_scores = motif_scorer.score_many(
        [
            (job['audio_path'], separation_result[0], separation_result[1])
            for job, separation_result in zip(batch_jobs, batch_separations)
        ],
        max_workers=MOTIF_WORKERS,
    )

    for job, separation_result, result in zip(batch_jobs, batch_separations, batch_scores):
        song = job['song']
        audio_path = job['audio_path']
        display_title = job['display_title']
        song_id = job['song_id']
        song_dir = job['song_dir']
        metadata_path = job['metadata_path']
        song_dir.mkdir(parents=True, exist_ok=True)
        stems, sample_rate, mixture = separation_result
        print(f'  [motif] {display_title}')
        if dataset_builder is None:
            save_song_stems(song, audio_path, stems, sample_rate)
        melodic = result['melodic_accompaniment']
        save_audio(song_dir / 'melodic_accompaniment.flac', melodic, sample_rate, audio_format='flac')

        for stale_candidate in song_dir.glob('candidate_*.flac'):
            stale_candidate.unlink()
        exported_candidates = [
            row for row in result['candidates'] if candidate_passes_export_filters(row)
        ]
        for row_index, row in enumerate(exported_candidates):
            start = round(row['start_sec'] * sample_rate)
            end = round(row['end_sec'] * sample_rate)
            filename = f"candidate_{row_index:03d}_{row['stem_name']}.flac"
            save_audio(song_dir / filename, melodic[:, start:end], sample_rate, audio_format='flac')
            row['candidate_file'] = filename
        for row in result['candidates']:
            all_rows.append({'song_id': song_id, 'title': display_title, **row})

        metadata = {
            'song_id': song_id, 'title': display_title, 'source_audio': str(audio_path),
            'sample_rate': sample_rate, 'similarity_mode': SIMILARITY_MODE,
            'candidates': result['candidates'],
        }
        metadata_path.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        selection_metadata = save_earliest_valid_selection(metadata_path, metadata)
        if dataset_builder is not None:
            if selection_metadata['status'] == 'selected':
                selected_candidate = selection_metadata['selections']['earliest']
                validated_variation = selected_candidate['onset_variation']
                dataset_result = dataset_builder.process_track(
                    song['track'],
                    audio_index,
                    separation_result=separation_result,
                    motif_scores=result,
                    validated_onset_variation=validated_variation,
                    selected_candidate=selected_candidate,
                )
            else:
                dataset_result = dataset_builder.record_rejection(
                    song['track'], selection_metadata.get('reason', 'final_motif_rejected')
                )
            dataset_results.append(dataset_result)
        del stems, mixture, result, melodic

failure_report_path = DIAGNOSTIC_DIR / 'audio_failures.jsonl'
failure_report_path.write_text(
    ''.join(json.dumps(failure, ensure_ascii=False) + '\n' for failure in audio_failures),
    encoding='utf-8',
)
if audio_failures:
    print(f'[audio failures] {len(audio_failures)}곡: {failure_report_path}')

scores_df = pd.DataFrame(all_rows)
scores_df.to_csv(DIAGNOSTIC_DIR / 'all_motif_scores.csv', index=False)
if scores_df.empty:
    display(scores_df)
else:
    display(scores_df.sort_values(['song_id', 'start_sec', 'stem_name']))
if dataset_results:
    from dataclasses import asdict
    display(pd.DataFrame([asdict(result) for result in dataset_results]))

accepted_track_ids = sorted({
    result.track_id
    for result in dataset_results
    if result.status in {'accepted', 'skipped'} and result.output_dir is not None
})
motif_version = {
    'schema': 'mohim_motif_dataset_v5',
    'data_source': DATA_SOURCE,
    'max_songs': MAX_SONGS,
    'separator_model': 'htdemucs_6s',
    'demucs_shifts': DEMUCS_SHIFTS,
    'bars': MOTIF_BARS,
    'search_seconds': MOTIF_SEARCH_SECONDS,
    'min_active_ratio': MIN_ACTIVE_RATIO,
    'max_onset_chroma_difference': MAX_ONSET_CHROMA_DIFFERENCE,
    'min_onset_similarity': MIN_ONSET_SIMILARITY,
    'min_pitch_class_span': MIN_PITCH_CLASS_SPAN,
    'min_mean_centered_similarity': MIN_MEAN_CENTERED_SIMILARITY,
    'min_onset_variation': MIN_ONSET_VARIATION,
    'similarity_mode': SIMILARITY_MODE,
    'accepted_track_ids': accepted_track_ids,
}
MOTIF_VERSION_PATH.write_text(
    json.dumps(motif_version, ensure_ascii=False, sort_keys=True, indent=2) + '\n',
    encoding='utf-8',
)
print('motif dataset version:', MOTIF_VERSION_PATH)

Streaming output truncated to the last 5000 lines.
[1647/2129] Rihanna - Love On The Brain
[1648/2129] Rihanna - Needed Me
[1649/2129] Rihanna - Only Girl (In The World)
[1650/2129] Rihanna - Rehab
[1651/2129] Rihanna - Rude Boy
[1652/2129] Rihanna - Russian Roulette
[1653/2129] Rihanna - Take A Bow
[1654/2129] Rihanna Featuring Jay-Z - Talk That Talk
[1655/2129] Rihanna - What Now
[1656/2129] Rihanna Featuring Drake - What's My Name?
[1657/2129] Rihanna - Where Have You Been
[1658/2129] Rihanna - You Da One
[1659/2129] Rita Ora - How We Do (Party)
[1660/2129] Rita Ora - I Will Never Let You Down
[1661/2129] Rita Ora - Let You Love Me
[1662/2129] Rita Ora Featuring Fatboy Slim - Praising You
[1663/2129] Ritt Momney - Put Your Records On
[1664/2129] Rixton - Me And My Broken Heart
[1665/2129] Rixton - Wait On Me
[1666/2129] Rob Thomas - Her Diamonds
[1667/2129] Rob Thomas - Someday
[1668/2129] Robin Schulz Featuring Francesco Yates - Sugar
[1669/2129] Robin Thicke Featuring Nicki Minaj 

,song_id,title,stem_name,candidate_index,start_sec,end_sec,active_ratio,onset_similarity,chroma_similarity,mean_centered_onset_similarity,mean_centered_chroma_similarity,similarity,pitch_class_span,candidate_file
12251,(G)I-DLE - I Do,(G)I-DLE - I Do,bass,0,0.34,8.979977,1.000000,0.674024,0.348757,0.238632,0.198198,0.226502,0.166667,NaN
12260,(G)I-DLE - I Do,(G)I-DLE - I Do,other,0,0.34,8.979977,0.998656,0.940318,0.740520,0.123101,0.309036,0.178881,0.166667,NaN
12252,(G)I-DLE - I Do,(G)I-DLE - I Do,bass,1,2.50,11.139977,1.000000,0.665192,0.362507,0.160030,0.214025,0.176229,0.250000,candidate_000_bass.flac
12261,(G)I-DLE - I Do,(G)I-DLE - I Do,other,1,2.50,11.139977,1.000000,0.942827,0.727999,0.093874,0.287934,0.152092,0.250000,candidate_002_other.flac
12253,(G)I-DLE - I Do,(G)I-DLE - I Do,bass,2,4.66,13.299977,1.000000,0.694816,0.389891,0.191108,0.253314,0.209770,0.166667,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32624,will.i.am Featuring Mick Jagger & Jennifer Lop...,will.i.am Featuring Mick Jagger & Jennifer Lop...,bass,7,25.06,34.020000,1.000000,0.456285,0.797375,0.206195,0.614480,0.328680,NaN,NaN
32625,will.i.am Featuring Mick Jagger & Jennifer Lop...,will.i.am Featuring Mick Jagger & Jennifer Lop...,bass,8,26.74,35.700000,1.000000,0.450001,0.588952,0.181251,0.200583,0.187050,NaN,NaN
32626,will.i.am Featuring Mick Jagger & Jennifer Lop...,will.i.am Featuring Mick Jagger & Jennifer Lop...,bass,9,27.30,36.260000,1.000000,0.519632,0.788147,0.275519,0.598617,0.372449,NaN,NaN
32627,will.i.am Featuring Mick Jagger & Jennifer Lop...,will.i.am Featuring Mick Jagger & Jennifer Lop...,bass,10,29.00,37.960000,1.000000,0.435017,0.570255,0.125077,0.167593,0.137832,NaN,NaN


,track_id,status,reason,output_dir,motif_stem
0,2-pistols-she-got-it,rejected,no_stem_candidate_passed_final_validation,None,None
1,21-savage-redrum,rejected,no_stem_candidate_passed_final_validation,None,None
2,2am-club-worry-about-you,rejected,ValueError: Selected motif mean-centered onset...,None,None
3,3-doors-down-its-not-my-time,rejected,no_stem_candidate_passed_final_validation,None,None
4,3oh-3-dont-trust-me-live,rejected,no_stem_candidate_passed_final_validation,None,None
...,...,...,...,...,...
2124,Zedd-maren-morris-and-beauz-make-you-say,accepted,None,/content/drive/MyDrive/MOHIM/motif_dataset_mea...,other
2125,Zedd-spectrum,rejected,no_stem_candidate_passed_final_validation,None,None
2126,Zendaya-replay,rejected,no_candidate_passed_export_filters,None,None
2127,Zendaya-something-new,accepted,None,/content/drive/MyDrive/MOHIM/motif_dataset_mea...,other


motif dataset version: /content/drive/MyDrive/MOHIM/motif_dataset_mean_centered/.mohim_motif_version


## 4. 현재 곡의 earliest motif 일괄 재생

현재 입력에 포함된 곡 중 저장된 earliest motif를 `DEBUG_TRACK_INDEX`부터 `DEBUG_TRACK_COUNT`곡씩 표시합니다. 예: 11~20곡은 `DEBUG_TRACK_INDEX = 10`으로 설정합니다.

In [ ]:
from IPython.display import Audio, display

DEBUG_TRACK_INDEX = 0
DEBUG_TRACK_COUNT = 10
selection_paths = []
for song in songs:
    selection_path = (
        DIAGNOSTIC_DIR / diagnostic_song_id(song) / 'motif_selections.json'
    )
    if selection_path.is_file():
        selection_metadata = json.loads(selection_path.read_text(encoding='utf-8'))
        if selection_metadata.get('status') == 'skipped':
            print(f'[skipped] {selection_metadata["title"]}: {selection_metadata["reason"]}')
            continue
        selection_paths.append(selection_path)
assert selection_paths, '현재 곡에 재생 가능한 motif selection이 없습니다.'
selected_paths = selection_paths[
    DEBUG_TRACK_INDEX:DEBUG_TRACK_INDEX + DEBUG_TRACK_COUNT
]
assert selected_paths, (
    f'재생할 곡이 없습니다: start={DEBUG_TRACK_INDEX}, total={len(selection_paths)}'
)

for track_number, selection_path in enumerate(
    selected_paths, start=DEBUG_TRACK_INDEX + 1
):
    selection_metadata = json.loads(selection_path.read_text(encoding='utf-8'))
    row = selection_metadata['selections']['earliest']
    print(f'\n{track_number}. {selection_metadata["title"]}')
    print({key: value for key, value in row.items() if key not in {'candidate_file', 'selection_file'}})
    display(Audio(filename=str(selection_path.parent / row['selection_file'])))

## 5. 선택 motif의 bar별 occurrence 판별

선택된 motif를 query로 삼아 `melodic_accompaniment.flac`의 모든 downbeat를 시작점 후보로 검사합니다. 비교와 최종 mask 구간은 downbeat 간격이 아니라 실제 `selected_earliest.flac` 길이를 사용합니다. 각 downbeat에 raw/mean-centered onset·chroma 점수와 threshold별 통과 여부를 기록하고, 겹치는 양성 window 중 최고점만 canonical occurrence로 남깁니다.

기존 motif 결과에서 occurrence만 볼 때는 0~2번 준비 셀까지만 실행하고 3~4번 추출 셀은 건너뛴 뒤 이 섹션을 실행합니다. `OCCURRENCE_TRACK_QUERY`에 artist/title/folder 일부를 넣으면 해당 곡만 불러옵니다.

In [ ]:
import json
import re
import time
from pathlib import Path
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from mohim.motif import _resize_time

OCCURRENCE_HOP_LENGTH = 512
OCCURRENCE_ONSET_N_FFT = 2048
OCCURRENCE_SILENCE_DB = -40.0
MIN_OCCURRENCE_ONSET_SIMILARITY = MIN_ONSET_SIMILARITY
MAX_OCCURRENCE_ONSET_CHROMA_DIFFERENCE = MAX_ONSET_CHROMA_DIFFERENCE
MIN_OCCURRENCE_MEAN_CENTERED_SIMILARITY = MIN_MEAN_CENTERED_SIMILARITY
MIN_OCCURRENCE_ACTIVE_RATIO = MIN_ACTIVE_RATIO
occurrence_analysis_cache = {}

def selected_motif_row(metadata):
    row = metadata.get('selections', {}).get('earliest')
    if row is None:
        raise KeyError('earliest motif selection이 없습니다.')
    return row

def occurrence_safe_folder_name(value):
    cleaned = re.sub(r'[\\/:*?"<>|\x00-\x1f]', '_', value)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip(' .')
    return cleaned[:120] or 'untitled'

assert 'songs' in globals() and 'song_paths' in globals() and song_paths, (
    '1번 입력 셀을 먼저 실행해 songs/song_paths를 준비하세요.'
)
assert len(songs) == len(song_paths), 'songs와 song_paths 길이가 다릅니다.'
selection_paths = []
for song, audio_path in zip(songs, song_paths):
    display_title = (
        f"{song['artist']} - {song['title']}" if song['artist'] else song['title']
    )
    base_name = occurrence_safe_folder_name(display_title)
    short_track_id = occurrence_safe_folder_name(str(song['track_id']))[:8]
    candidates = (
        DIAGNOSTIC_DIR / base_name / 'motif_selections.json',
        DIAGNOSTIC_DIR / f'{base_name[:109]} [{short_track_id}]' / 'motif_selections.json',
    )
    selected_path = None
    for selection_path in candidates:
        if not selection_path.is_file():
            continue
        try:
            metadata = json.loads(selection_path.read_text(encoding='utf-8'))
            source_path = Path(metadata['source_audio'])
            if source_path.resolve() != Path(audio_path).resolve():
                continue
            if metadata.get('status') != 'selected':
                continue
            selection = selected_motif_row(metadata)
            required_paths = (
                selection_path.parent / selection['selection_file'],
                selection_path.parent / 'melodic_accompaniment.flac',
                source_path,
            )
            if all(path.is_file() for path in required_paths):
                selected_path = selection_path
                break
        except (OSError, ValueError, KeyError, TypeError):
            continue
    if selected_path is None:
        print(f'[missing occurrence input] {display_title}')
    else:
        selection_paths.append(selected_path)
assert selection_paths, '현재 song_paths에 occurrence 검수 가능한 selected motif가 없습니다.'
print(f'[occurrence inputs] {len(selection_paths)}곡 / selected songs {len(song_paths)}곡')

def occurrence_shifted_cosine(first, second, *, mean_center):
    scores = []
    for shift in range(-3, 4):
        if shift < 0:
            first_overlap, second_overlap = first[-shift:], second[:shift]
        elif shift > 0:
            first_overlap, second_overlap = first[:-shift], second[shift:]
        else:
            first_overlap, second_overlap = first, second
        if mean_center:
            first_overlap = first_overlap - np.mean(first_overlap)
            second_overlap = second_overlap - np.mean(second_overlap)
        scores.append(float(cosine_similarity(
            first_overlap[None], second_overlap[None]
        )[0, 0]))
    return max(scores)

def occurrence_aligned_chroma_similarities(
    query_raw, query_centered, candidate_raw, candidate_centered,
):
    best = None
    for pitch_shift in range(12):
        shifted_raw = np.roll(candidate_raw, pitch_shift, axis=0)
        shifted_centered = np.roll(candidate_centered, pitch_shift, axis=0)
        for time_shift in (-1, 0, 1):
            if time_shift < 0:
                query_raw_overlap = query_raw[:, :time_shift]
                candidate_raw_overlap = shifted_raw[:, -time_shift:]
                query_centered_overlap = query_centered[:, :time_shift]
                candidate_centered_overlap = shifted_centered[:, -time_shift:]
            elif time_shift > 0:
                query_raw_overlap = query_raw[:, time_shift:]
                candidate_raw_overlap = shifted_raw[:, :-time_shift]
                query_centered_overlap = query_centered[:, time_shift:]
                candidate_centered_overlap = shifted_centered[:, :-time_shift]
            else:
                query_raw_overlap = query_raw
                candidate_raw_overlap = shifted_raw
                query_centered_overlap = query_centered
                candidate_centered_overlap = shifted_centered
            raw_similarity = float(cosine_similarity(
                query_raw_overlap.reshape(1, -1),
                candidate_raw_overlap.reshape(1, -1),
            )[0, 0])
            centered_similarity = float(cosine_similarity(
                query_centered_overlap.reshape(1, -1),
                candidate_centered_overlap.reshape(1, -1),
            )[0, 0])
            result = {
                'raw_similarity': raw_similarity,
                'centered_similarity': centered_similarity,
                'pitch_shift': pitch_shift if pitch_shift <= 6 else pitch_shift - 12,
                'time_shift': time_shift,
            }
            if best is None or (centered_similarity, raw_similarity) > (
                best['centered_similarity'], best['raw_similarity'],
            ):
                best = result
    return best

def occurrence_feature_tracks(audio, sample_rate):
    mono = np.asarray(audio, dtype=np.float32).reshape(-1)
    preroll_frames = OCCURRENCE_ONSET_N_FFT // OCCURRENCE_HOP_LENGTH
    padded = np.pad(mono, (preroll_frames * OCCURRENCE_HOP_LENGTH, 0))
    padded_onset = librosa.onset.onset_strength(
        y=padded, sr=sample_rate, hop_length=OCCURRENCE_HOP_LENGTH,
        n_fft=OCCURRENCE_ONSET_N_FFT,
    )
    rms = librosa.feature.rms(y=mono, hop_length=OCCURRENCE_HOP_LENGTH)[0]
    onset = padded_onset[preroll_frames:preroll_frames + len(rms)]
    chroma = librosa.feature.chroma_cens(
        y=mono, sr=sample_rate, hop_length=OCCURRENCE_HOP_LENGTH
    )
    frame_count = min(len(onset), chroma.shape[1], len(rms))
    rms_db = librosa.amplitude_to_db(rms[:frame_count] + 1e-8, ref=1.0)
    return onset[:frame_count], chroma[:, :frame_count], rms_db

def occurrence_window_features(onset, chroma, rms_db, start_frame, end_frame):
    end_frame = min(end_frame, len(onset), chroma.shape[1], len(rms_db))
    if end_frame - start_frame < 2:
        raise ValueError('occurrence window가 두 frame보다 짧습니다.')
    onset_window = _resize_time(
        onset[None, start_frame:end_frame], 256
    ).ravel()
    raw_chroma = _resize_time(chroma[:, start_frame:end_frame], 64)
    centered_chroma = raw_chroma - np.mean(
        raw_chroma, axis=0, keepdims=True
    )
    active = rms_db[start_frame:end_frame] > OCCURRENCE_SILENCE_DB
    active_ratio = float(np.mean(active)) if len(active) else 0.0
    return (
        onset_window, raw_chroma, centered_chroma, active_ratio
    )

def classify_occurrence_downbeats(
    scores, *, min_onset=MIN_OCCURRENCE_ONSET_SIMILARITY,
    max_difference=MAX_OCCURRENCE_ONSET_CHROMA_DIFFERENCE,
    min_mean_centered=MIN_OCCURRENCE_MEAN_CENTERED_SIMILARITY,
    min_active_ratio=MIN_OCCURRENCE_ACTIVE_RATIO, forced_start_sec=None,
):
    classified = scores.copy()
    classified['active_ratio_pass'] = (
        classified['active_ratio'] >= min_active_ratio
    )
    classified['onset_similarity_pass'] = (
        classified['onset_similarity'] >= min_onset
    )
    classified['onset_chroma_difference_pass'] = (
        classified['onset_chroma_difference'] <= max_difference
    )
    classified['mean_centered_onset_pass'] = (
        classified['mean_centered_onset_similarity'] >= min_mean_centered
    )
    classified['mean_centered_chroma_pass'] = (
        classified['mean_centered_chroma_similarity'] >= min_mean_centered
    )
    pass_columns = [
        'active_ratio_pass', 'onset_similarity_pass',
        'onset_chroma_difference_pass', 'mean_centered_onset_pass',
        'mean_centered_chroma_pass',
    ]
    classified['is_occurrence_candidate'] = classified[pass_columns].all(axis=1)
    classified['is_occurrence'] = classified['is_occurrence_candidate']
    if forced_start_sec is not None:
        forced_index = (
            classified['start_sec'] - float(forced_start_sec)
        ).abs().idxmin()
        classified.at[forced_index, 'is_occurrence'] = True
    return classified

def fill_short_occurrence_gaps(occurrences, *, max_gap_seconds):
    max_gap_seconds = float(max_gap_seconds)
    if max_gap_seconds < 0:
        raise ValueError('max_gap_seconds는 0 이상이어야 합니다.')
    intervals = sorted(
        (
            {
                'start_sec': float(item['start_sec']),
                'end_sec': float(item['end_sec']),
            }
            for item in occurrences
        ),
        key=lambda item: item['start_sec'],
    )
    filled = []
    for interval in intervals:
        if not filled:
            filled.append(interval)
            continue
        gap_seconds = interval['start_sec'] - filled[-1]['end_sec']
        if gap_seconds <= max_gap_seconds:
            filled[-1]['end_sec'] = max(
                filled[-1]['end_sec'], interval['end_sec']
            )
        else:
            filled.append(interval)
    return filled

def rasterize_occurrence_mask(occurrences, *, duration_seconds, frame_count):
    frame_count = int(frame_count)
    if duration_seconds <= 0 or frame_count < 1:
        raise ValueError('duration_seconds와 frame_count는 양수여야 합니다.')
    mask = np.zeros(int(frame_count), dtype=np.float32)
    frames_per_second = frame_count / float(duration_seconds)
    for occurrence in occurrences:
        start = max(0, int(np.floor(
            float(occurrence['start_sec']) * frames_per_second
        )))
        end = min(frame_count, int(np.ceil(
            float(occurrence['end_sec']) * frames_per_second
        )))
        if end > start:
            mask[start:end] = 1.0
    return mask

def json_records(frame):
    return [
        {
            key: value.item() if isinstance(value, np.generic) else value
            for key, value in row.items()
        }
        for row in frame.to_dict(orient='records')
    ]

def occurrence_thresholds():
    return {
        'min_onset_similarity': MIN_OCCURRENCE_ONSET_SIMILARITY,
        'max_onset_chroma_difference':
            MAX_OCCURRENCE_ONSET_CHROMA_DIFFERENCE,
        'min_mean_centered_similarity':
            MIN_OCCURRENCE_MEAN_CENTERED_SIMILARITY,
        'min_active_ratio': MIN_OCCURRENCE_ACTIVE_RATIO,
    }

def compute_motif_occurrence_scores(selection_path, *, force=False):
    selection_path = Path(selection_path)
    cache_key = str(selection_path.resolve())
    if not force and cache_key in occurrence_analysis_cache:
        return occurrence_analysis_cache[cache_key]
    metadata = json.loads(selection_path.read_text(encoding='utf-8'))
    selection = selected_motif_row(metadata)
    song_dir = selection_path.parent
    motif_path = song_dir / selection['selection_file']
    melodic_path = song_dir / 'melodic_accompaniment.flac'
    source_path = Path(metadata['source_audio'])
    for required_path in (motif_path, melodic_path, source_path):
        if not required_path.is_file():
            raise FileNotFoundError(required_path)
    melodic, sample_rate = librosa.load(melodic_path, sr=None, mono=True)
    motif, _ = librosa.load(motif_path, sr=sample_rate, mono=True)
    motif_duration = len(motif) / sample_rate
    full_onset, full_chroma, full_rms_db = occurrence_feature_tracks(
        melodic, sample_rate
    )
    motif_onset, motif_chroma, motif_rms_db = occurrence_feature_tracks(
        motif, sample_rate
    )
    query = occurrence_window_features(
        motif_onset, motif_chroma, motif_rms_db,
        0, min(len(motif_onset), motif_chroma.shape[1], len(motif_rms_db)),
    )
    _, downbeats = beat_tracker(str(source_path))
    downbeats = np.asarray(downbeats, dtype=np.float64).reshape(-1)
    if not len(downbeats):
        raise ValueError(f'downbeat가 없습니다: {metadata["title"]}')
    rows = []
    audio_duration = len(melodic) / sample_rate
    for downbeat_index, downbeat in enumerate(downbeats):
        start_sec = float(downbeat)
        end_sec = start_sec + motif_duration
        if start_sec < 0 or end_sec > audio_duration:
            continue
        start_frame = int(librosa.time_to_frames(
            start_sec, sr=sample_rate, hop_length=OCCURRENCE_HOP_LENGTH
        ))
        end_frame = int(librosa.time_to_frames(
            end_sec, sr=sample_rate, hop_length=OCCURRENCE_HOP_LENGTH
        ))
        try:
            window = occurrence_window_features(
                full_onset, full_chroma, full_rms_db, start_frame, end_frame
            )
        except ValueError:
            continue
        raw_onset = occurrence_shifted_cosine(
            query[0], window[0], mean_center=False
        )
        centered_onset = occurrence_shifted_cosine(
            query[0], window[0], mean_center=True
        )
        chroma_alignment = occurrence_aligned_chroma_similarities(
            query[1], query[2], window[1], window[2]
        )
        raw_chroma = chroma_alignment['raw_similarity']
        centered_chroma = chroma_alignment['centered_similarity']
        rows.append({
            'downbeat_index': downbeat_index,
            'downbeat_number': downbeat_index + 1,
            'start_sec': start_sec,
            'end_sec': end_sec,
            'active_ratio': window[3],
            'onset_similarity': raw_onset,
            'chroma_similarity': raw_chroma,
            'onset_chroma_difference': abs(raw_onset - raw_chroma),
            'mean_centered_onset_similarity': centered_onset,
            'mean_centered_chroma_similarity': centered_chroma,
            'chroma_pitch_shift': chroma_alignment['pitch_shift'],
            'chroma_time_shift': chroma_alignment['time_shift'],
        })
    raw_scores = pd.DataFrame(rows)
    if raw_scores.empty:
        raise ValueError(f'비교할 bar window가 없습니다: {metadata["title"]}')
    anchor_start = float(selection['start_sec'])
    anchor_index = (raw_scores['start_sec'] - anchor_start).abs().idxmin()
    raw_scores['is_selected_anchor'] = False
    raw_scores.at[anchor_index, 'is_selected_anchor'] = True
    scores = classify_occurrence_downbeats(
        raw_scores, forced_start_sec=anchor_start
    )
    occurrence_rows = scores[scores['is_occurrence']].sort_values('start_sec')
    motif_intervals = [
        {
            'downbeat_index': int(row['downbeat_index']),
            'start_sec': float(row['start_sec']),
            'end_sec': float(row['end_sec']),
            'is_selected_anchor': bool(row['is_selected_anchor']),
        }
        for _, row in occurrence_rows.iterrows()
    ]
    max_mask_gap_seconds = 2.0 * motif_duration
    mask_intervals = fill_short_occurrence_gaps(
        motif_intervals, max_gap_seconds=max_mask_gap_seconds
    )
    mask_path = song_dir / 'motif_occurrence_mask.npy'
    occurrence_mask = rasterize_occurrence_mask(
        mask_intervals, duration_seconds=audio_duration,
        frame_count=len(full_onset),
    )
    np.save(mask_path, occurrence_mask)
    output_path = song_dir / 'motif_occurrences.json'
    payload = {
        'schema': 'mohim_motif_occurrence_mask_v2',
        'song_id': metadata['song_id'],
        'title': metadata['title'],
        'source_audio': metadata['source_audio'],
        'selection_file': selection['selection_file'],
        'motif_start_sec': anchor_start,
        'motif_end_sec': anchor_start + motif_duration,
        'motif_duration_seconds': motif_duration,
        'duration_seconds': audio_duration,
        'motif_bars': MOTIF_BARS,
        'mask_semantics': (
            '1 inside detected motif intervals and internal gaps no longer than '
            '2x motif duration, 0 elsewhere'
        ),
        'mask_fill_max_gap_seconds': max_mask_gap_seconds,
        'thresholds': occurrence_thresholds(),
        'downbeat_candidate_count': len(scores),
        'occurrence_candidate_count': int(scores['is_occurrence_candidate'].sum()),
        'occurrence_count': int(scores['is_occurrence'].sum()),
        'downbeat_scores': json_records(scores),
        'motif_intervals': motif_intervals,
        'mask_intervals': mask_intervals,
        'mask_file': mask_path.name,
        'mask_frame_count': len(occurrence_mask),
        'mask_sample_rate': sample_rate,
        'mask_hop_length': OCCURRENCE_HOP_LENGTH,
        'mask_frame_rate': sample_rate / OCCURRENCE_HOP_LENGTH,
    }
    output_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    assert bool(scores.loc[anchor_index, 'is_occurrence'])
    result = {
        'metadata': metadata, 'selection': selection, 'scores': scores,
        'motif_path': motif_path, 'melodic_path': melodic_path,
        'source_path': source_path, 'output_path': output_path,
        'mask_path': mask_path, 'occurrence_mask': occurrence_mask,
        'motif_duration': motif_duration, 'audio_duration': audio_duration,
        'motif_intervals': motif_intervals, 'mask_intervals': mask_intervals,
    }
    occurrence_analysis_cache[cache_key] = result
    return result

def format_duration(seconds):
    minutes, seconds = divmod(max(0.0, float(seconds)), 60.0)
    return f'{int(minutes):02d}:{seconds:04.1f}'

def compute_all_motif_occurrences(paths, *, force=False):
    started_at = time.monotonic()
    all_occurrence_rows = []
    failures = []
    total = len(paths)
    for index, selection_path in enumerate(paths, 1):
        elapsed = time.monotonic() - started_at
        eta = elapsed / (index - 1) * (total - index + 1) if index > 1 else None
        eta_text = format_duration(eta) if eta is not None else '계산 중'
        print(f'[occurrence {index}/{total}] ETA {eta_text}: {selection_path.parent.name}')
        try:
            analysis = compute_motif_occurrence_scores(
                selection_path, force=force
            )
        except Exception as error:
            failures.append({
                'selection_path': str(selection_path),
                'error_type': type(error).__name__,
                'error_message': str(error),
            })
            print(f'  [failed] {type(error).__name__}: {error}')
            continue
        rows = analysis['scores'].copy()
        rows.insert(0, 'title', analysis['metadata']['title'])
        rows.insert(0, 'song_id', analysis['metadata']['song_id'])
        all_occurrence_rows.append(rows)
    occurrence_scores_df = (
        pd.concat(all_occurrence_rows, ignore_index=True)
        if all_occurrence_rows else pd.DataFrame()
    )
    if not occurrence_scores_df.empty:
        occurrence_scores_df.to_csv(
            DIAGNOSTIC_DIR / 'all_motif_occurrence_scores.csv', index=False
        )
    failure_path = DIAGNOSTIC_DIR / 'occurrence_failures.jsonl'
    failure_path.write_text(
        ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in failures),
        encoding='utf-8',
    )
    elapsed = time.monotonic() - started_at
    print(
        f'[occurrence complete] {total - len(failures)}/{total}곡, '
        f'실패 {len(failures)}곡, elapsed {format_duration(elapsed)}'
    )
    return occurrence_scores_df, pd.DataFrame(failures)

RUN_OCCURRENCE_BATCH = False
if RUN_OCCURRENCE_BATCH:
    occurrence_scores_df, occurrence_failures_df = compute_all_motif_occurrences(
        selection_paths
    )
    display(occurrence_scores_df)
    display(occurrence_failures_df)
else:
    print('전체 곡 occurrence 저장은 RUN_OCCURRENCE_BATCH = True로 바꾸고 실행하세요.')

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML

def format_timestamp(seconds):
    minutes, seconds = divmod(float(seconds), 60.0)
    return f'{int(minutes):02d}:{seconds:05.2f}'

def review_motif_occurrences(
    track_index, min_onset, max_difference,
    min_mean_centered, min_active_ratio, audio_count,
):
    analysis = compute_motif_occurrence_scores(selection_paths[int(track_index)])
    scores = classify_occurrence_downbeats(
        analysis['scores'],
        min_onset=min_onset,
        max_difference=max_difference, min_mean_centered=min_mean_centered,
        min_active_ratio=min_active_ratio,
        forced_start_sec=float(analysis['selection']['start_sec']),
    )
    candidates = scores[scores['is_occurrence_candidate']]
    occurrences = scores[scores['is_occurrence']].sort_values('start_sec')
    anchor = float(analysis['selection']['start_sec'])
    display(HTML(f'<h3>{analysis["metadata"]["title"]}</h3>'))
    print(
        f'anchor={format_timestamp(anchor)}, motif length='
        f'{analysis["motif_duration"]:.2f}s, downbeats={len(scores)}, '
        f'candidates={len(candidates)}, occurrences={len(occurrences)}'
    )
    print(f'default 판정 저장 위치: {analysis["output_path"]}')
    display(HTML('<b>Selected 4-bar motif</b>'))
    display(Audio(filename=str(analysis['motif_path'])))
    display(HTML('<b>Original full song</b>'))
    display(Audio(filename=str(analysis['source_path'])))

    preview_rate = 10
    preview_intervals = fill_short_occurrence_gaps(
        json_records(occurrences),
        max_gap_seconds=2.0 * analysis['motif_duration'],
    )
    preview_mask = rasterize_occurrence_mask(
        preview_intervals, duration_seconds=analysis['audio_duration'],
        frame_count=max(1, int(np.ceil(analysis['audio_duration'] * preview_rate))),
    )
    preview_times = np.arange(len(preview_mask)) / preview_rate
    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
    axes[0].plot(scores['start_sec'], scores['onset_similarity'], label='raw onset')
    axes[0].plot(scores['start_sec'], scores['chroma_similarity'], label='raw chroma')
    axes[0].axhline(min_onset, color='tab:blue', linestyle=':', label='min onset')
    axes[1].plot(
        scores['start_sec'], scores['mean_centered_onset_similarity'],
        label='mean-centered onset', alpha=0.8,
    )
    axes[1].plot(
        scores['start_sec'], scores['mean_centered_chroma_similarity'],
        label='mean-centered chroma', alpha=0.8,
    )
    axes[2].fill_between(
        preview_times, preview_mask, step='post', color='limegreen', alpha=0.65
    )
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].set_ylabel('motif mask')
    for axis in axes[:2]:
        axis.axvline(anchor, color='red', linestyle='--', linewidth=2)
        for _, row in candidates.iterrows():
            axis.axvspan(row['start_sec'], row['end_sec'], color='gold', alpha=0.08)
        for _, row in occurrences.iterrows():
            axis.axvspan(row['start_sec'], row['end_sec'], color='limegreen', alpha=0.18)
        axis.set_ylim(-0.2, 1.05)
        axis.grid(alpha=0.2)
        axis.legend(loc='lower right')
    axes[0].set_ylabel('raw cosine')
    axes[1].set_ylabel('mean-centered cosine')
    axes[2].set_xlabel('song time (seconds)')
    plt.tight_layout()
    plt.show()

    table = scores.copy()
    table['status'] = np.select(
        [table['is_occurrence'], table['is_occurrence_candidate']],
        ['OCCURRENCE', 'candidate'], default='',
    )
    table['start'] = table['start_sec'].map(format_timestamp)
    table['end'] = table['end_sec'].map(format_timestamp)
    table_columns = [
        'downbeat_number', 'status', 'is_selected_anchor', 'start', 'end', 'active_ratio',
        'onset_similarity', 'chroma_similarity', 'onset_chroma_difference',
        'mean_centered_onset_similarity',
        'mean_centered_chroma_similarity',
    ]
    display(table[table_columns].style.format({
        column: '{:.3f}'
        for column in table_columns
        if column not in {'downbeat_number', 'status', 'is_selected_anchor', 'start', 'end'}
    }))

    if occurrences.empty or audio_count == 0:
        print('재생할 canonical occurrence가 없습니다. threshold를 조정하세요.')
        return
    full_mix, full_sr = librosa.load(analysis['source_path'], sr=None, mono=True)
    melodic, melodic_sr = librosa.load(analysis['melodic_path'], sr=None, mono=True)
    for number, (_, occurrence) in enumerate(
        occurrences.head(int(audio_count)).iterrows(), 1
    ):
        clip_start = max(0.0, float(occurrence['start_sec']) - 1.0)
        clip_end = float(occurrence['end_sec']) + 1.0
        display(HTML(
            f'<h4>{number}. downbeat {int(occurrence["downbeat_number"])} | '
            f'{format_timestamp(occurrence["start_sec"])} | '
            f'score={occurrence["similarity"]:.3f}</h4>'
        ))
        print('full mix (+/- 1s context)')
        display(Audio(
            full_mix[int(clip_start * full_sr):int(clip_end * full_sr)], rate=full_sr
        ))
        print('melodic accompaniment (+/- 1s context)')
        display(Audio(
            melodic[int(clip_start * melodic_sr):int(clip_end * melodic_sr)],
            rate=melodic_sr,
        ))

track_options = []
for index, path in enumerate(selection_paths):
    metadata = json.loads(path.read_text(encoding='utf-8'))
    track_options.append((f'{index + 1:04d} | {metadata["title"]}', index))
track_widget = widgets.Dropdown(
    options=track_options, description='곡:', layout=widgets.Layout(width='90%')
)
onset_widget = widgets.FloatSlider(
    value=MIN_OCCURRENCE_ONSET_SIMILARITY, min=0.0, max=1.0, step=0.01,
    description='raw onset:', continuous_update=False,
)
difference_widget = widgets.FloatSlider(
    value=MAX_OCCURRENCE_ONSET_CHROMA_DIFFERENCE, min=0.0, max=1.0, step=0.01,
    description='raw 차이:', continuous_update=False,
)
centered_widget = widgets.FloatSlider(
    value=MIN_OCCURRENCE_MEAN_CENTERED_SIMILARITY, min=-0.1, max=1.0, step=0.01,
    description='centered:', continuous_update=False,
)
active_widget = widgets.FloatSlider(
    value=MIN_OCCURRENCE_ACTIVE_RATIO, min=0.0, max=1.0, step=0.01,
    description='active:', continuous_update=False,
)
audio_widget = widgets.IntSlider(
    value=4, min=0, max=20, step=1, description='재생 개수:',
    continuous_update=False,
)
review_output = widgets.interactive_output(
    review_motif_occurrences,
    {
        'track_index': track_widget,
        'min_onset': onset_widget, 'max_difference': difference_widget,
        'min_mean_centered': centered_widget, 'min_active_ratio': active_widget,
        'audio_count': audio_widget,
    },
)
display(widgets.VBox([
    track_widget, onset_widget, difference_widget, centered_widget,
    active_widget, audio_widget,
]), review_output)